In [ ]:
# Run 008. Save Version → Save & Run All trên Kaggle GPU, Internet On.
from pathlib import Path
import sys, subprocess, json, importlib, importlib.util, os, time, shutil
import pandas as pd
import numpy as np
from IPython.display import display, FileLink, clear_output

SOURCE_COMMIT = "eecd36c3178d93dfe473dbcbfa5e362e04168664"
BASE = Path(os.environ.get('QUANTSTREAM_RUN_BASE', '/kaggle/working' if Path('/kaggle/working').exists() else str(Path.cwd()/'notebook_work')))
BASE.mkdir(parents=True,exist_ok=True)
CODE = Path(os.environ.get('QUANTSTREAM_SOURCE_DIR',str(BASE/('quantstream-source-'+SOURCE_COMMIT[:12]))))
WORK = BASE/'QuantStream_Run008'
WORK.mkdir(parents=True,exist_ok=True)
N_TRIALS = 8
MAX_EPOCHS = 96
MIN_EPOCHS = 24
JOB_SECONDS = 10800
NOTEBOOK_URL = ''
assert N_TRIALS == 8 and 1 <= MIN_EPOCHS <= MAX_EPOCHS <= 120 and 60 <= JOB_SECONDS <= 14400
if not CODE.exists():
    subprocess.check_call(['git','clone','-q','--no-checkout','https://github.com/taiiswibu/quantstream-crypto-research.git',str(CODE)])
subprocess.check_call(['git','-C',str(CODE),'checkout','-q','--detach',SOURCE_COMMIT])
assert subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
assert not subprocess.check_output(['git','-C',str(CODE),'status','--porcelain'],text=True).strip()
if 'quantstream' in sys.modules and Path(sys.modules['quantstream'].__file__).resolve().parent != CODE/'quantstream':
    raise RuntimeError('Restart Session: kernel đã import quantstream từ source khác')
sys.path.insert(0,str(CODE))
modules={'optuna':'optuna>=4.5,<6','pyarrow':'pyarrow>=17,<27','xgboost':'xgboost>=2.1,<4',
         'sklearn':'scikit-learn>=1.5,<2','matplotlib':'matplotlib>=3.9,<4','filelock':'filelock>=3.16,<5'}
missing=[package for name,package in modules.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--only-binary=:all:',*missing])
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if Path('/kaggle/working').exists() and DEVICE != 'cuda':
    raise RuntimeError('Chọn GPU trong Kaggle Settings trước khi chạy lượt train đầy đủ')
from quantstream.common import write_json, sha256
from quantstream.extended_training import prepare_dataset, find_parent_snapshot
from quantstream.features import FEATURES, fold_data
from quantstream.workflow_plots import dataset_figures, scaler_figure, training_figures, evaluation_figures, architecture_figure
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(11,4),'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
write_json(WORK/'notebook-run.json',{'notebook_url':NOTEBOOK_URL or None,'source_commit':SOURCE_COMMIT,
           'torch_loaded':torch.__version__,'numpy_loaded':np.__version__,'gpu':(torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'),
           'trials':N_TRIALS,'epochs':MAX_EPOCHS,'whole_job_seconds':JOB_SECONDS})
print('Source:',SOURCE_COMMIT,'GPU:',(torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'))

In [ ]:
# Thu thập 2023-01 → 2026-11-13 UTC; checksum archive và nến đã đóng.
parent = Path(os.environ['QUANTSTREAM_PARENT_SNAPSHOT']) if os.environ.get('QUANTSTREAM_PARENT_SNAPSHOT') else find_parent_snapshot(Path('/kaggle/input'),BASE/'snapshot-cache')
print('Snapshot:',str(parent) if parent else 'Collect the same frozen period')
manifest = prepare_dataset(WORK,parent)
bars = pd.read_parquet(WORK/'data/processed/bars.parquet')
quality = json.loads((WORK/'data/processed/data_quality.json').read_text())
display(bars.groupby('symbol').agg(rows=('timestamp','size'),first=('timestamp','min'),last=('timestamp','max')))
display(pd.DataFrame([{k:v for k,v in quality.items() if k!='gaps'}]))
display(pd.DataFrame(quality['gaps']))
print('Source:',manifest['provider'],'Synthetic:',manifest['synthetic'])

In [ ]:
# Làm sạch, missingness và tính hợp lệ OHLCV; không điền giá qua khoảng trống.
from quantstream.data import VALUES
missing = bars[VALUES].isna().sum()
fig,axes = plt.subplots(1,2,figsize=(12,4))
axes[0].bar(missing.index,missing.values,color='#0f766e')
axes[0].set(title='Missing cells after validation',ylabel='Cells');axes[0].tick_params(axis='x',rotation=75)
gap_counts={symbol:sum(r['missing_bars'] for r in quality['gaps'] if r['symbol']==symbol) for symbol in bars.symbol.unique()}
axes[1].bar(gap_counts.keys(),gap_counts.values(),color='#b45309')
axes[1].set(title='Missing hourly candles',ylabel='Candles')
for ax in axes: ax.grid(axis='y',alpha=.2)
plt.tight_layout()
(WORK/'figures').mkdir(exist_ok=True)
fig.savefig(WORK/'figures/01-data-quality.png',dpi=150,bbox_inches='tight');plt.show()
assert np.isfinite(bars[VALUES]).all().all()
assert not bars.duplicated(['symbol','timestamp']).any()
print('Quarantine:',quality['quarantine_rows'],'Duplicate rows:',quality['duplicate_rows'])

In [ ]:
# Feature chỉ dùng quá khứ; nhãn return giữ 4 giờ sau trễ vào 1 giờ.
features = pd.read_parquet(WORK/'data/processed/features.parquet')
split = manifest['split']
display(features[['symbol','decision_time','entry_time','label_end_time','target_return',*FEATURES[:4]]].head(10))
display(pd.DataFrame(split['tuning_folds']))
print('Retrospective test:',split['test_start'],'→',split['end'])
print('Các giai đoạn lịch sử đã được quan sát. Không gọi đây là test đầu tư độc lập.')

In [ ]:
# EDA và ranh giới thời gian; đồ thị mô tả không được dùng fit scaler.
dataset_figures(WORK)

In [ ]:
# Fit scaler trên train của fold đầu; không fit trên validation/test.
train, validation = fold_data(features,split['tuning_folds'][0]['validation_start'],split['tuning_folds'][0]['validation_end'])
assert train.label_end_time.max() < validation.decision_time.min()
scaler_stats = scaler_figure(train, WORK)
display(scaler_stats)
print('Đồ thị demo scaler neural ddof=1. Ridge dùng StandardScaler ddof=0; scaler thực của từng fit nằm trong ZIP. XGBoost dùng feature gốc.')

In [ ]:
# Kiến trúc thật của hai họ neural trong giao thức cố định.
from quantstream.final_research import conditions
from quantstream.search import SequenceRegressor
neural=[p for p in conditions(MAX_EPOCHS,MIN_EPOCHS) if p['family'] in ['tcn','gru'] and p['target_transform']=='none']
architecture=[]
fig,axes=plt.subplots(1,2,figsize=(12,6))
for ax,params in zip(axes,neural):
    model=SequenceRegressor(params)
    architecture.append({'family':params['family'],'parameters':sum(p.numel() for p in model.parameters()),
        'activation':'SiLU' if params['family']=='tcn' else 'sigmoid/tanh gates','context':48,'features':18,'width':32})
    encoder='4 causal residual blocks; SiLU; dilations 1, 2, 4, 8' if params['family']=='tcn' else 'GRU; sigmoid gates and tanh state'
    labels=['48 hourly steps × 18 features',encoder,'Last hidden state: 32','Dropout 0.15 → Linear(32, 1)']
    for i,label in enumerate(labels):
        y=.85-i*.21
        ax.text(.5,y,label,ha='center',va='center',wrap=True,fontsize=9,
            bbox={'boxstyle':'round,pad=1','facecolor':'#ecfdf5','edgecolor':'#0f766e'})
        if i<3: ax.annotate('',xy=(.5,y-.15),xytext=(.5,y-.06),arrowprops={'arrowstyle':'->','color':'#64748b'})
    ax.set(xlim=(0,1),ylim=(0,1),title=params['family'].upper());ax.axis('off')
fig.suptitle('Fixed neural architectures · output units restored before evaluation')
plt.tight_layout();fig.savefig(WORK/'figures/06-architecture.png',dpi=150,bbox_inches='tight');plt.show()
display(pd.DataFrame(architecture))
print(SequenceRegressor(neural[0]))
del model

In [ ]:
# Tám điều kiện cân bằng; sau đó kiểm tra một finalist mỗi họ trên ba seed.
plan_path=WORK/'training_plan.json'
status=json.loads((WORK/'status.json').read_text()) if (WORK/'status.json').exists() else {}
if plan_path.exists():
    old=json.loads(plan_path.read_text())
    assert old['protocol']=='run008-final-v1' and old['source']['commit']==SOURCE_COMMIT
    assert old['dataset_manifest_sha256']==sha256(WORK/'dataset_manifest.json')
    assert (old['epochs'],old['minimum_epochs'],old['seconds'])==(MAX_EPOCHS,MIN_EPOCHS,JOB_SECONDS)
    if status.get('status')!='completed': raise RuntimeError('Giữ evidence của lượt chưa hoàn tất; không xóa để chạy lại chọn điểm')
    print('Đọc lượt đã hoàn tất; không train lại.')
else:
    log=WORK/'training-console.log'
    environment={**os.environ,'PYTHONPATH':str(CODE)+os.pathsep+os.environ.get('PYTHONPATH','')}
    started=time.monotonic()
    try:
        with log.open('w') as stream:
            worker=subprocess.Popen([sys.executable,'-m','quantstream.final_research','--root',str(WORK),
                '--epochs',str(MAX_EPOCHS),'--minimum-epochs',str(MIN_EPOCHS),'--seconds',str(JOB_SECONDS),'--device',DEVICE],
                cwd=str(CODE),env=environment,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
            while worker.poll() is None:
                if time.monotonic()-started > JOB_SECONDS+15:
                    import signal
                    os.killpg(worker.pid,signal.SIGTERM) if os.name=='posix' else worker.terminate()
                    try: worker.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        os.killpg(worker.pid,signal.SIGKILL) if os.name=='posix' else worker.kill()
                        worker.wait()
                    raise TimeoutError('Dừng bởi watchdog; giữ partial evidence')
                clear_output(wait=True)
                print('Runtime:',DEVICE,'Elapsed:',round(time.monotonic()-started),'/',JOB_SECONDS,'seconds')
                if (WORK/'progress.json').exists(): print(json.loads((WORK/'progress.json').read_text()))
                print(''.join(log.read_text(errors='replace').splitlines(keepends=True)[-8:]))
                time.sleep(5)
            if worker.returncode: raise RuntimeError('Xem error.txt và training-console.log trong ZIP')
    finally:
        if 'worker' in locals() and worker.poll() is None:
            worker.terminate()
            try: worker.wait(timeout=10)
            except subprocess.TimeoutExpired: worker.kill();worker.wait()
        evidence=shutil.make_archive(str(BASE/'QuantStream_Run008_Output'),'zip',WORK)
        display(FileLink(evidence))
status=json.loads((WORK/'status.json').read_text())
display(pd.DataFrame([status]))

In [ ]:
# Giữ mọi điều kiện, cả model chưa vượt baseline.
display(pd.read_csv(WORK/'trials.csv'))
from quantstream.final_plots import figures
figures(WORK)
# Learning rate ghi trong lịch sử, không phải đường minh họa.
fig,ax=plt.subplots(figsize=(11,4))
for trial in pd.read_csv(WORK/'trials.csv').itertuples():
    path=WORK/'trials'/f'{trial.number:03}'/'fold-0/early-stop/history.csv'
    if path.exists():
        h=pd.read_csv(path)
        ax.plot(h.epoch,h.learning_rate,marker='.',label=f'{trial.params_family}/{trial.params_target_transform}')
ax.set(title='Recorded learning rate · first fold',xlabel='Epoch',ylabel='Learning rate');ax.legend();ax.grid(alpha=.2)
plt.tight_layout();fig.savefig(WORK/'figures/19-learning-rates.png',dpi=150,bbox_inches='tight');plt.show()


In [ ]:
# Quyết định chỉ dùng development; khoảng tuần là chẩn đoán xấp xỉ.
selection=json.loads((WORK/'selection.json').read_text())
display(pd.DataFrame(selection['shortlist']))
print('Best research configuration:',selection['params'])
print('Recommended:',selection['recommended_model'],'Signal approved:',selection['signal_approved'])
print('Validation đã dùng nghiên cứu nhiều lần; không phải bằng chứng đầu tư độc lập.')

In [ ]:
# Sai số dự báo và equity là hai câu hỏi khác nhau.
assert status['status']=='completed'
metrics=evaluation_figures(WORK)
display(metrics)
display(pd.read_json(WORK/'retrospective_asset_metrics.json'))
strategies=json.loads((WORK/'retrospective_strategies.json').read_text())
display(pd.DataFrame([{'strategy':k,**v} for k,v in strategies.items()]).drop(columns='note',errors='ignore'))

In [ ]:
# Model, scaler, feature contract và forecast parity để mang về local.
DEPLOYMENT=WORK/'deployment'
bundle=json.loads((DEPLOYMENT/'bundle_manifest.json').read_text())
for name,expected in bundle['files'].items():
    assert sha256(DEPLOYMENT/name)==expected,name
exported=json.loads((DEPLOYMENT/'metadata.json').read_text())
display(pd.DataFrame([{k:exported[k] for k in ['model','training_rows','selected_epochs','created_at','sha256','deployment_status']}]))
print('Model và cấu hình:',exported['hyperparameters'])
print('Import không train lại. Parity kiểm tra cùng đầu vào, scaler và đơn vị return.')
print('Artifact role:',exported.get('artifact_role'))
print('Training last label:',exported.get('training_last_label'))
print('Independent test of these refit weights:',exported.get('test'))
print('Model chưa đạt gate vẫn ghi dự báo; app không nhận lệnh paper mới.')

In [ ]:
# Gói nhỏ để nhập local; giữ riêng checkpoint đánh giá và weights refit mới.
from quantstream.final_plots import diagnostic_zip
evidence=shutil.make_archive(str(BASE/'QuantStream_Run008_Output'),'zip',WORK)
model_zip=shutil.make_archive(str(BASE/'QuantStream_Run008_Model'),'zip',DEPLOYMENT)
audit_zip=shutil.make_archive(str(BASE/'QuantStream_Run008_Audit_Model'),'zip',WORK/'audit_model/deployment')
diagnostics=diagnostic_zip(WORK,BASE/'QuantStream_Run008_Diagnostics.zip')
for file in [model_zip,audit_zip,str(diagnostics),evidence]: display(FileLink(file))
print('Local: nạp Model.zip. Điểm hồi cứu chỉ thuộc Audit_Model.zip, không gán cho weights refit mới.')